# ML-Ready EO Data Preparation using openEO

Training machine learning models on Earth Observation data requires large volumes of labelled samples with meaningful spectral and spatial features. Traditionally this involves downloading raw imagery, running preprocessing pipelines locally, and managing large intermediate files - a significant barrier for many users.

**openEO** removes this barrier by running all heavy computation in the cloud, directly on the data archive. You define a processing graph, openEO executes it server-side, and you download only the compact feature table or image chips you need for training.

---

## Workflow

```mermaid
flowchart TD
    GT["Ground truth labels\n(LUCAS 2022 parquet)"]

    GT --> CDSE

    subgraph CDSE["openEO on CDSE"]
        direction TB
        L["load Sentinel-2 L2A"]
        CM["cloud masking (SCL dilation)"]
        TC["temporal median composite"]
        UDF(apply custom process)
        IDX["spectral indices using built-in processes"]
        L --> CM --> TC --> UDF & IDX
    end

    CDSE --> AGG["aggregate_spatial"]
    CDSE --> PATCH["sample_by_feature"]

    AGG --> FT["Point features"]
    PATCH --> IC["Polygon patch outputs"]
```

---

## Notebook structure

| Part | Approach | Output | ML use |
|---|---|---|---|
| **A** | Point extraction | Feature table (CSV) | Random Forest, XGBoost |
| **B** | Polygon-based sampling | One raster per polygon | CNN, ViT, U-Net |

---

## Prerequisites

- Free [CDSE account](https://dataspace.copernicus.eu)
- Python: `openeo`, `geopandas`, `scikit-image`, `scikit-learn`
- Ground truth parquet is downloaded automatically from the CloudFerro URL in Section 2


In [15]:
import openeo
import geopandas as gpd
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from shapely.geometry import box
from openeo.extra.spectral_indices import compute_indices
from sklearn.model_selection import train_test_split

In [ ]:
conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
conn

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## 2 · Ground Truth — LUCAS 2022

We use the [LUCAS Copernicus 2022](https://data.jrc.ec.europa.eu/dataset/e3fe3cd0-44db-470e-8769-172a8b9e8874) survey as labelled ground truth.  
The notebook downloads the GeoParquet file, filters it to **Belgium**, and prepares both point- and polygon-based train/test splits from the same source data.

> **Adapting to your use case:** replace the parquet URL and bounding box with your own labelled dataset. Any GeoDataFrame with a `geometry` column and a `target` column works.


In [ ]:
LUCAS_PQ_PATH = Path("LUCAS_2022_4326.parquet")
BELGIUM_BBOX = (2.5, 49.4, 6.5, 51.6)  # west, south, east, north

lucas = gpd.read_parquet(LUCAS_PQ_PATH)
west_b, south_b, east_b, north_b = BELGIUM_BBOX
lucas = lucas.cx[west_b:east_b, south_b:north_b].copy()

print(f"Loaded {len(lucas)} LUCAS polygons in Belgium  |  CRS: {lucas.crs}")
lucas.head(3)

Loaded 5567 LUCAS polygons in Belgium  |  CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "e

,point_id,user_id,point_nuts0,pi_extension,point_ex_ante,point_lat,point_long,point_altitude,point_copernicus,point_grassland,...,lu1_code,poly_area_sqm,surveycprncando,surveycprnlc1n,surveycprnclc1e,surveycprnclc1w,surveycprnclc1s,surveycprnlc,col_hex,geometry
765,38082954,FRSU236,FR,0,0,49.47599,2.910148,39,1,0,...,U120,3901.746495,1,17,21,40,51,C10,80ff00,"POLYGON ((2.90979 49.47575, 2.90978 49.47575, ..."
767,37883044,FRSU209,FR,0,0,50.26284,2.510591,141,1,0,...,U111,953.396090,1,32,6,10,6,B16,ffff00,"POLYGON ((2.51221 50.26185, 2.51221 50.26185, ..."
792,37943010,FRSU215,FR,0,0,49.96439,2.641608,80,1,0,...,U111,8208.436826,1,51,51,51,51,B11,ffd300,"POLYGON ((2.64112 49.96471, 2.64114 49.96472, ..."


In [23]:
# Detect LC1 column — LUCAS 2022 uses 'survey_lc1'
lc1_col = next((c for c in lucas.columns if c.lower() in ("lc1", "survey_lc1")), None)
if lc1_col is None:
    raise ValueError(f"No LC1 column found. Available: {list(lucas.columns)}")

groundtruth = lucas[["point_id", lc1_col, "geometry"]].copy()
groundtruth["geometry"] = groundtruth["geometry"].apply(lambda x: x.centroid)
groundtruth["target"]   = groundtruth[lc1_col].apply(lambda x: ord(x[0]) - 65)
# work on a subset
counts = groundtruth["target"].value_counts()

top10_classes = counts[counts >= 50].head().index

groundtruth_balanced = (
    groundtruth[groundtruth["target"].isin(top10_classes)]
    .groupby("target")
    .sample(n=25, random_state=42)
    .reset_index(drop=True)
)

groundtruth_train, groundtruth_test = train_test_split(groundtruth_balanced, test_size=0.25, random_state=333)
print(f"Train: {len(groundtruth_train)}  |  Test: {len(groundtruth_test)}")
print(f"{len(groundtruth_balanced)}, {groundtruth_balanced['target'].nunique()}, {groundtruth_balanced['target'].value_counts()}")

Train: 93  |  Test: 32
125, 5, target
0    25
1    25
2    25
4    25
5    25
Name: count, dtype: int64


## 3 · Part A — Point-based Feature Extraction

Extracts a **flat feature vector** per sample point using `aggregate_spatial`.  
The result is a JSON file that maps each sample to its feature values — directly loadable as a pandas DataFrame for model training.


In [24]:
BANDS   = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
INDICES = ["NDVI", "EVI", "NDWI"]
TEXTURE_BANDS = ["contrast", "variance", "NDFI", "brightness"]

west, south, east, north = groundtruth_train.total_bounds
aoi = {"west": float(west), "south": float(south), "east": float(east), "north": float(north)}
temporal_extent = ["2022-06-01", "2022-10-31"]

In [6]:
# Load + cloud-mask Sentinel-2
s2  = conn.load_collection("SENTINEL2_L2A", temporal_extent=temporal_extent,
                            spatial_extent=aoi, bands=BANDS, max_cloud_cover=80)
scl = conn.load_collection("SENTINEL2_L2A", temporal_extent=temporal_extent,
                            spatial_extent=aoi, bands=["SCL"], max_cloud_cover=80)
cloud_mask = scl.process("to_scl_dilation_mask", data=scl,
                          kernel1_size=17, kernel2_size=77,
                          mask1_values=[2, 4, 5, 6, 7],
                          mask2_values=[3, 8, 9, 10, 11],
                          erosion_kernel_size=3)
s2_masked = s2.mask(cloud_mask)

In [7]:
# Temporal median composite 
composite    = s2_masked.reduce_dimension(reducer="median", dimension="t")

In [8]:
indices_cube = compute_indices(composite, INDICES)

# Texture UDF runs on the spatial tile after temporal compositing.
texture_udf = openeo.UDF.from_file("udf.py")
texture_cube = composite.apply_neighborhood(
    process=texture_udf,
    size=[
        {"dimension": "x", "value": 128, "unit": "px"},
        {"dimension": "y", "value": 128, "unit": "px"},
    ],
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
    context={"padding_window_size": 33},
)
feature_cube = composite.merge_cubes(texture_cube).merge_cubes(indices_cube)

In [ ]:
# aggregate_spatial: one feature vector per sample point
point_geojson = json.loads(groundtruth_train.set_geometry("geometry").to_json())
agg_cube = feature_cube.aggregate_spatial(point_geojson, reducer="median")

In [10]:
OUTPUT_POINTS = "point_based_features.csv"

# Submit batch job to CDSE 
agg_cube.execute_batch(title="point extraction", outputfile=OUTPUT_POINTS)
print(f"Saved → {OUTPUT_POINTS}")

0:00:00 Job 'j-260812081434440ea8746ab626bb0306': send 'start'
0:00:02 Job 'j-260812081434440ea8746ab626bb0306': queued (progress 0%)
0:00:07 Job 'j-260812081434440ea8746ab626bb0306': queued (progress 0%)
0:00:14 Job 'j-260812081434440ea8746ab626bb0306': queued (progress 0%)
0:00:22 Job 'j-260812081434440ea8746ab626bb0306': queued (progress 0%)
0:00:32 Job 'j-260812081434440ea8746ab626bb0306': queued (progress 0%)
0:00:44 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:01:00 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:01:19 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:01:43 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:02:13 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:02:51 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:03:38 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A)
0:04:36 Job 'j-260812081434440ea8746ab626bb0306': running (progress N/A

## 4 · Part B — Patch-based Feature Extraction

Extracts one image output per training polygon using `sample_by_feature=True`.  
This version reuses the polygon split prepared in Section 2, so there is no second data-loading block here.

> If you want fixed-size chips centred on points instead of polygon footprints, reintroduce a centroid-plus-buffer geometry step here.

In [ ]:
PATCH_SIZE_PX = 64
BUFFER_M      = (PATCH_SIZE_PX * 10) / 2   # 320 m at 10 m/px

patches = groundtruth_train.to_crs(32631).copy()
patches["geometry"] = patches.geometry.centroid.buffer(BUFFER_M, cap_style=3)
patches = patches.to_crs(4326)
patches["patch_id"] = patches["point_id"].astype(str)
patches

In [ ]:
west_p, south_p, east_p, north_p = patches.total_bounds
patch_aoi = {"west": float(west_p), "south": float(south_p),
             "east": float(east_p),  "north": float(north_p)}

In [9]:
s2_p  = conn.load_collection("SENTINEL2_L2A", temporal_extent=temporal_extent,
                              spatial_extent=patch_aoi, bands=BANDS, max_cloud_cover=80)
scl_p = conn.load_collection("SENTINEL2_L2A", temporal_extent=temporal_extent,
                              spatial_extent=patch_aoi, bands=["SCL"], max_cloud_cover=80)
cm_p  = scl_p.process("to_scl_dilation_mask", data=scl_p,
                        kernel1_size=17, kernel2_size=77,
                        mask1_values=[2, 4, 5, 6, 7],
                        mask2_values=[3, 8, 9, 10, 11],
                        erosion_kernel_size=3)

composite_p   = s2_p.mask(cm_p).reduce_dimension(reducer="median", dimension="t")

In [10]:
texture_p     = composite_p.apply_neighborhood(
    process=openeo.UDF.from_file("udf.py"),
    size=[{"dimension": "x", "value": 128, "unit": "px"},
          {"dimension": "y", "value": 128, "unit": "px"}],
    overlap=[{"dimension": "x", "value": 32, "unit": "px"},
             {"dimension": "y", "value": 32, "unit": "px"}],
    context={"padding_window_size": 33},
)
feature_patch = composite_p.merge_cubes(texture_p).merge_cubes(
    compute_indices(composite_p, INDICES)
)

In [11]:
patch_cube = feature_patch.filter_spatial(json.loads(polygon_groundtruth_train.to_json()))

In [ ]:
OUTPUT_PATCH_DIR = "patch_chips"
os.makedirs(OUTPUT_PATCH_DIR, exist_ok=True)

# Export one GeoTIFF per training polygon.
job = patch_cube.create_job(
    out_format="GTiff",
    title="patch extraction",
    sample_by_feature=True,
    feature_id_property="patch_id",
    job_options={"driver-memory": "2G", "executor-memory": "2G", "max-executors": 10},
)
job.start_and_wait()
job.get_results().download_files(OUTPUT_PATCH_DIR)

print(f"Saved polygon-based patch outputs in {OUTPUT_PATCH_DIR}")

0:00:00 Job 'j-26081209094445459e31234cf8d9331d': send 'start'
0:00:07 Job 'j-26081209094445459e31234cf8d9331d': queued (progress 0%)
0:00:13 Job 'j-26081209094445459e31234cf8d9331d': queued (progress 0%)
0:00:21 Job 'j-26081209094445459e31234cf8d9331d': queued (progress 0%)
0:00:31 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:00:43 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:00:57 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:01:14 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:01:35 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:02:01 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:02:33 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:03:12 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:04:01 Job 'j-26081209094445459e31234cf8d9331d': running (progress N/A)
0:05:01 Job 'j-26081209094445459e31234cf8d9331d': running (progress